# Carevora Emotional Wellness – Training Notebook
Targets: **`high_risk_label`** (binary) and **`trajectory_label`** (improving / stable / deteriorating)

Plan:
1. Load data (resident-level split කලින්ම කරලා තියෙනවා)
2. Leakage-free features + longitudinal (past-visits-only) features
3. Baselines → Optuna tuning (GroupKFold, resident leak නැතුව) → ensemble
4. Threshold validation set එකෙන් තෝරනවා, **test set එක අන්තිමට එක පාරයි** පාවිච්චි කරන්නේ
5. Metrics: Accuracy, Precision, Recall, F1, ROC-AUC, confusion matrix

> ⚠️ Data synthetic. Score එකෙන් මැනෙන්නේ model එක simulator එක ඉගෙන ගත්ත විදිහ මිසක් real-world accuracy නෙවෙයි.

In [ ]:
!pip -q install optuna lightgbm
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, optuna, zipfile, os, glob
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, classification_report)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
import lightgbm as lgb
optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED = 42; np.random.seed(SEED)

## 1. Data load (Google Drive eken)

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

# Zip eka Drive eke thiyena path eka methana denna (nathnam auto-search karanawa)
ZIP_PATH = '/content/drive/MyDrive/carevora_synthetic_dataset_v2.zip'

if not os.path.exists(ZIP_PATH):
    found = glob.glob('/content/drive/MyDrive/**/carevora*v2*.zip', recursive=True)
    assert found, "Drive eke carevora*v2*.zip file eka hambune nehe. ZIP_PATH eka hariyata denna."
    ZIP_PATH = found[0]
print('Using:', ZIP_PATH)

zipfile.ZipFile(ZIP_PATH).extractall('/content/data')
D = os.path.dirname(glob.glob('/content/data/**/train.csv', recursive=True)[0])
tr, va, te = [pd.read_csv(f'{D}/{n}.csv') for n in ['train','validation','test']]
print(tr.shape, va.shape, te.shape)
print('Resident overlap:', len(set(tr.resident_id)&set(va.resident_id)), len(set(tr.resident_id)&set(te.resident_id)))
tr.head()

## 2. Features
- Leakage columns drop: `current_risk_score_0_1`, `current_risk_level`, `trajectory_label`, ids, split
- Longitudinal features **past visits විතරයි** පාවිච්චි කරනවා (lag1, expanding mean, rolling-3, current − personal history) → ඉදිරි visits වලින් leak වෙන්නේ නෑ

In [ ]:
DROP = ['resident_id','split','current_risk_score_0_1','current_risk_level','trajectory_label','high_risk_label']
BASE = [c for c in tr.columns if c not in DROP]
CORE = [c for c in BASE if c != 'visit_month']

def add_features(d):
    d = d.sort_values(['resident_id','visit_month']).copy()
    g = d.groupby('resident_id')
    new = {}
    for c in CORE:
        new[c+'_lag1']  = g[c].shift(1)
        new[c+'_lag2']  = g[c].shift(2)
        new[c+'_expmean'] = g[c].transform(lambda s: s.expanding().mean().shift(1))
        new[c+'_expstd']  = g[c].transform(lambda s: s.expanding().std().shift(1))
        new[c+'_roll3'] = g[c].transform(lambda s: s.rolling(3, min_periods=1).mean())
        new[c+'_vs_hist'] = d[c] - new[c+'_expmean']
        new[c+'_delta1']  = d[c] - new[c+'_lag1']
    d = pd.concat([d, pd.DataFrame(new, index=d.index)], axis=1)
    # composite indicators
    d['neg_affect']   = d.facial_sad_prob + d.vocal_stress_score - d.facial_happy_prob - d.vocal_positive_score
    d['social_deficit'] = d.questionnaire_loneliness_score_0_10 + d.social_isolation_score_0_10 - d.activity_participation_score_0_10
    d['wellness_gap'] = d.questionnaire_loneliness_score_0_10 - d.emotional_wellness_score_0_10
    d['contact_total'] = d.family_contacts_per_week + d.social_interactions_per_week
    return d

tr, va, te = [add_features(x).reset_index(drop=True) for x in (tr, va, te)]
FULL = [c for c in tr.columns if c not in DROP]
print(len(BASE), 'base features |', len(FULL), 'full features')

## 3. Helper functions

In [ ]:
def metrics(y, p, proba):
    return dict(Accuracy=accuracy_score(y,p), Precision=precision_score(y,p,average='binary' if len(set(y))==2 else 'macro',zero_division=0),
                Recall=recall_score(y,p,average='binary' if len(set(y))==2 else 'macro'),
                F1=f1_score(y,p,average='binary' if len(set(y))==2 else 'macro'),
                ROC_AUC=roc_auc_score(y,proba) if len(set(y))==2 else roc_auc_score(y,proba,multi_class='ovr',average='macro'))

def best_threshold(y, proba, metric='f1'):
    ts = np.linspace(0.2, 0.8, 121)
    sc = [f1_score(y, proba>=t) if metric=='f1' else accuracy_score(y, proba>=t) for t in ts]
    return ts[int(np.argmax(sc))]

def groupcv_score(make_model, X, y, groups, n=5, multi=False):
    oof = np.zeros((len(y), 3)) if multi else np.zeros(len(y))
    for a, b in GroupKFold(n).split(X, y, groups):
        m = make_model(); m.fit(X.iloc[a], y.iloc[a])
        oof[b] = m.predict_proba(X.iloc[b]) if multi else m.predict_proba(X.iloc[b])[:,1]
    return oof

## 4. TASK A – `high_risk_label`
### 4.1 Baselines (validation set)

In [ ]:
ya_tr, ya_va, ya_te = tr.high_risk_label, va.high_risk_label, te.high_risk_label
print('Majority-class baseline acc (val):', max(ya_va.mean(), 1-ya_va.mean()).round(4))

def baselines(cols):
    return {
      'LogReg': make_pipeline(SimpleImputer(), StandardScaler(), LogisticRegression(C=0.5, max_iter=3000)),
      'RandForest': make_pipeline(SimpleImputer(), RandomForestClassifier(500, min_samples_leaf=5, n_jobs=-1, random_state=SEED)),
      'HistGB': HistGradientBoostingClassifier(max_iter=250, learning_rate=0.04, max_depth=3, random_state=SEED),
      'LightGBM': lgb.LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=8, subsample=0.8, subsample_freq=1,
                                     colsample_bytree=0.7, reg_lambda=5, verbose=-1, random_state=SEED)}
rows = []
for fs_name, cols in [('base', BASE), ('base+longitudinal', FULL)]:
    for n, m in baselines(cols).items():
        m.fit(tr[cols], ya_tr); pr = m.predict_proba(va[cols])[:,1]
        rows.append({'features': fs_name, 'model': n, **metrics(ya_va, pr>=0.5, pr)})
pd.DataFrame(rows).round(4).sort_values('F1', ascending=False)

### 4.2 Optuna tuning (train GroupKFold CV – validation/test touch කරන්නේ නෑ)

In [ ]:
Xtr, Xva, Xte = tr[FULL], va[FULL], te[FULL]
groups = tr.resident_id

def obj_lgb(trial):
    p = dict(n_estimators=trial.suggest_int("n_estimators",100,600), learning_rate=trial.suggest_float("lr",0.01,0.1,log=True),
             num_leaves=trial.suggest_int("leaves",4,24), min_child_samples=trial.suggest_int("mcs",20,100),
             subsample=trial.suggest_float("ss",0.6,1), subsample_freq=1, colsample_bytree=trial.suggest_float("cs",0.3,0.9),
             reg_lambda=trial.suggest_float("l2",1e-2,50,log=True), verbose=-1, random_state=SEED)
    oof = groupcv_score(lambda: lgb.LGBMClassifier(**p), Xtr, ya_tr, groups)
    return roc_auc_score(ya_tr, oof)
st = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
st.optimize(obj_lgb, n_trials=40, show_progress_bar=True)
print('best CV AUC', round(st.best_value,4)); best_lgb = st.best_params

In [ ]:
def obj_lr(trial):
    C = trial.suggest_float('C', 1e-3, 10, log=True)
    mk = lambda: make_pipeline(SimpleImputer(), StandardScaler(), LogisticRegression(C=C, max_iter=3000))
    return roc_auc_score(ya_tr, groupcv_score(mk, Xtr, ya_tr, groups))
st2 = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
st2.optimize(obj_lr, n_trials=20); best_C = st2.best_params['C']; print('LR C =', best_C, 'CV AUC', round(st2.best_value,4))

### 4.3 Ensemble + threshold (validation) + FINAL test

In [ ]:
lp = best_lgb
def mk_lgb(seed=SEED):
    return lgb.LGBMClassifier(n_estimators=lp['n_estimators'], learning_rate=lp['lr'], num_leaves=lp['leaves'], min_child_samples=lp['mcs'],
                              subsample=lp['ss'], subsample_freq=1, colsample_bytree=lp['cs'], reg_lambda=lp['l2'], verbose=-1, random_state=seed)
models = {'lgb': mk_lgb(), 'lgb2': mk_lgb(7),
          'lr': make_pipeline(SimpleImputer(), StandardScaler(), LogisticRegression(C=best_C, max_iter=3000)),
          'hgb': HistGradientBoostingClassifier(max_iter=250, learning_rate=0.04, max_depth=3, random_state=SEED)}
val_p = {}
for n, m in models.items():
    m.fit(Xtr, ya_tr); val_p[n] = m.predict_proba(Xva)[:,1]
    print(f'{n:5s} val AUC {roc_auc_score(ya_va, val_p[n]):.4f}')
ens_va = np.mean(list(val_p.values()), axis=0)
thr = best_threshold(ya_va, ens_va, 'f1')
print('Ensemble val AUC', round(roc_auc_score(ya_va, ens_va),4), '| chosen threshold (val F1):', round(thr,3))
print(pd.Series(metrics(ya_va, ens_va>=thr, ens_va)).round(4))

In [ ]:
# Final: train+val ඒකට refit කරලා test එකේ එක පාරක් විතරක් evaluate
trva = pd.concat([tr, va]).reset_index(drop=True)
Xall, yall = trva[FULL], trva.high_risk_label
te_p = []
for n, m in models.items():
    m.fit(Xall, yall); te_p.append(m.predict_proba(Xte)[:,1])
ens_te = np.mean(te_p, axis=0)
res = metrics(ya_te, ens_te>=thr, ens_te)
print('=== TEST (high_risk_label) ==='); print(pd.Series(res).round(4))
print(classification_report(ya_te, ens_te>=thr, target_names=['not high','high']))
ConfusionMatrixDisplay(confusion_matrix(ya_te, ens_te>=thr), display_labels=['not high','high']).plot(cmap='Blues'); plt.show()

### 4.4 Feature importance

In [ ]:
m = models['lgb']; m.fit(Xall, yall)
imp = pd.Series(m.booster_.feature_importance('gain'), index=FULL).sort_values(ascending=False).head(20)
imp[::-1].plot.barh(figsize=(7,6), title='Top 20 features (LightGBM gain)'); plt.tight_layout(); plt.show()

## 5. TASK B – `trajectory_label` (3 classes)
Chance level ≈ 33%. Features: base + longitudinal.

In [ ]:
cls = ['deteriorating','improving','stable']; cmap = {c:i for i,c in enumerate(cls)}
yb_tr, yb_va, yb_te = [d.trajectory_label.map(cmap) for d in (tr, va, te)]
print('Majority baseline (val):', yb_va.value_counts(normalize=True).max().round(4))

def obj_b(trial):
    p = dict(n_estimators=trial.suggest_int('n',100,500), learning_rate=trial.suggest_float('lr',0.01,0.1,log=True),
             num_leaves=trial.suggest_int('leaves',4,16), min_child_samples=trial.suggest_int('mcs',20,100),
             subsample=trial.suggest_float('ss',0.6,1), subsample_freq=1, colsample_bytree=trial.suggest_float('cs',0.3,0.9),
             reg_lambda=trial.suggest_float('l2',1e-2,50,log=True), verbose=-1, random_state=SEED)
    oof = groupcv_score(lambda: lgb.LGBMClassifier(**p), Xtr, yb_tr, groups, multi=True)
    return f1_score(yb_tr, oof.argmax(1), average='macro')
sb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
sb.optimize(obj_b, n_trials=30, show_progress_bar=True); bp = sb.best_params; print('best CV macro-F1', round(sb.best_value,4))

In [ ]:
def mk_b(seed=SEED):
    return lgb.LGBMClassifier(n_estimators=bp['n'], learning_rate=bp['lr'], num_leaves=bp['leaves'], min_child_samples=bp['mcs'],
                              subsample=bp['ss'], subsample_freq=1, colsample_bytree=bp['cs'], reg_lambda=bp['l2'], verbose=-1, random_state=seed)
mb = [mk_b(), mk_b(7), make_pipeline(SimpleImputer(), StandardScaler(), LogisticRegression(C=0.1, max_iter=3000))]
pv = np.mean([m.fit(Xtr, yb_tr).predict_proba(Xva) for m in mb], axis=0)
print('VAL:', pd.Series(metrics(yb_va, pv.argmax(1), pv)).round(4).to_dict())

Xall_b = pd.concat([Xtr, Xva]); yall_b = pd.concat([yb_tr, yb_va])
pt = np.mean([m.fit(Xall_b, yall_b).predict_proba(Xte) for m in mb], axis=0)
print('=== TEST (trajectory_label) ==='); print(pd.Series(metrics(yb_te, pt.argmax(1), pt)).round(4))
print(classification_report(yb_te, pt.argmax(1), target_names=cls))
ConfusionMatrixDisplay(confusion_matrix(yb_te, pt.argmax(1)), display_labels=cls).plot(cmap='Blues', xticks_rotation=20); plt.show()

## 6. Save models
(Colab එකේ download කරන්න)

In [ ]:
import joblib
joblib.dump({'models': models, 'features': FULL, 'threshold': thr}, 'carevora_high_risk_model.joblib')
joblib.dump({'models': mb, 'features': FULL, 'classes': cls}, 'carevora_trajectory_model.joblib')
files.download('carevora_high_risk_model.joblib'); files.download('carevora_trajectory_model.joblib')